# Revised Linear Model

Implementing our Logistic Regression with scikit-learn methods, rather than manually

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from data_cleaning import cleaned_data

In [3]:
data = cleaned_data()

# TODO: remove 'VOTEHOW'
X = pd.get_dummies(data.drop(['VOTED', 'VOTEHOW'], axis=1), drop_first=True)
feature_order = X.columns
X = X.to_numpy()

voted_map = {"not_voted": 1, "voted": 0}
y = data['VOTED'].map(voted_map).to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1001)

# Appending biases
X_train = np.concatenate((np.ones((X_train.shape[0], 1)), X_train), axis=1)
X_test = np.concatenate((np.ones((X_test.shape[0], 1)), X_test), axis=1)

In [7]:
# Check if the arrays are created correctly.

print(X_train.shape)
print(X_train[:5])
print(feature_order)

(50270, 16)
[[1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [1. 1. 0. 1. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
 [1. 1. 0. 1. 1. 0. 0. 0. 1. 0. 0. 0. 1. 0. 1. 0.]
 [1. 0. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 0.]
 [1. 0. 1. 0. 1. 0. 0. 0. 0. 1. 0. 0. 1. 0. 0. 0.]]
Index(['AGE_middle_age', 'AGE_elderly', 'SEX_female', 'RACE_black',
       'RACE_indian_aleut_eskimo', 'RACE_asian', 'RACE_others',
       'EDUC_college_grad', 'EDUC_master_higher', 'NATIVITY_foreign_born',
       'REGION_midwest', 'REGION_south', 'REGION_west', 'FAMINC_middle',
       'FAMINC_upper'],
      dtype='str')


Logistic regression with regularization

In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate 

def logreg_variations(X, y, l1_ratio = 0, C = 1.0, cv = 5, scoring = None):
    logreg = LogisticRegression(l1_ratio = l1_ratio, C = C, solver = 'liblinear')
    scores = cross_validate(logreg, X, y, cv=cv, return_train_score=True, scoring = scoring)

    return scores

In [9]:
# get the scores for L1 and L2
l1_version = logreg_variations(X_train, y_train, l1_ratio = 1)
l2_version = logreg_variations(X_train, y_train, l1_ratio = 0)

# calculate the mean
l1_mean_train_score = np.mean(l1_version["train_score"])
l1_mean_test_score =  np.mean(l1_version["test_score"])

l2_mean_train_score = np.mean(l2_version["train_score"])
l2_mean_test_score =  np.mean(l2_version["test_score"])

print(f'For L1 the mean cross validation train score is {l1_mean_train_score} and the mean test score is {l1_mean_train_score}')
print(f'For L2 the mean cross validation train score is {l2_mean_train_score} and the mean test score is {l2_mean_train_score}')

For L1 the mean cross validation train score is 0.771513825343147 and the mean test score is 0.771513825343147
For L2 the mean cross validation train score is 0.7715187984881638 and the mean test score is 0.7715187984881638


Finding a value for "C"

In [10]:
C_list = [1.0, 0.1, 0.01, 0.001, 0.0001]
for C in C_list:
    scores = logreg_variations(X_train, y_train, l1_ratio = 0, C = C)
    mean_train_score = np.mean(scores["train_score"])
    mean_test_score =  np.mean(scores["test_score"])
    print(f'With C={C} the cross validation mean train score is {mean_train_score} and the mean test score is {mean_test_score}')

With C=1.0 the cross validation mean train score is 0.7715187984881638 and the mean test score is 0.7709966182613887
With C=0.1 the cross validation mean train score is 0.7715287447781978 and the mean test score is 0.771135866321862
With C=0.01 the cross validation mean train score is 0.7701710761885817 and the mean test score is 0.7700616669982097
With C=0.001 the cross validation mean train score is 0.7667992838671176 and the mean test score is 0.7667992838671176
With C=0.0001 the cross validation mean train score is 0.7667992838671176 and the mean test score is 0.7667992838671176


In [11]:
for C in C_list:
    logreg = LogisticRegression(l1_ratio = 0, C = C, solver = 'liblinear')
    logreg.fit(X_train, y_train)
    # organise coefficients and features into a table
    coef_table = pd.DataFrame(zip(feature_order, np.transpose(logreg.coef_)), columns=['features', 'coef'])
    print("C:", C)
    print(coef_table)

C: 1.0
                    features                     coef
0             AGE_middle_age  [-0.041520216994342656]
1                AGE_elderly    [-0.5877211537205577]
2                 SEX_female    [-1.2535127205034502]
3                 RACE_black   [-0.14695020834523506]
4   RACE_indian_aleut_eskimo  [-0.011319166293085683]
5                 RACE_asian     [0.5939907856216023]
6                RACE_others    [0.24495170127699367]
7          EDUC_college_grad    [0.26085295674637127]
8         EDUC_master_higher    [-0.4788627261228515]
9      NATIVITY_foreign_born     [-1.419409492365885]
10            REGION_midwest     [0.5889417081660858]
11              REGION_south    [0.11633186817304735]
12               REGION_west    [0.21088300365242604]
13             FAMINC_middle    [0.12672334521735126]
14              FAMINC_upper     [-0.719494235359778]
C: 0.1
                    features                     coef
0             AGE_middle_age  [-0.045370068073000425]
1             

Logistic regression with different types of scores

In [12]:
# Accuracy provides the highest model score

scorers_list = ['accuracy', 'precision','f1', 'recall']

for scorer in scorers_list:
    scores = logreg_variations(X_train, y_train, l1_ratio = 0, C = 0.1, scoring = scorer)
    mean_train_score = np.mean(scores["train_score"])
    mean_test_score =  np.mean(scores["test_score"])
    print(f'With {scorer} the cross validation mean train score is {mean_train_score} and the mean test score is {mean_test_score}')

With accuracy the cross validation mean train score is 0.7715287447781978 and the mean test score is 0.771135866321862
With precision the cross validation mean train score is 0.5397508966066886 and the mean test score is 0.5366000672059654
With f1 the cross validation mean train score is 0.219116945493134 and the mean test score is 0.21773192601325828
With recall the cross validation mean train score is 0.1374860518742827 and the mean test score is 0.13665296142398684


Cross validation of all hyperparameters: L1/L2, C, scoring

In [8]:
import itertools

l1_ratios = [0, 1]
C_list = [1.0, 0.1, 0.01, 0.001, 0.0001]
scorers_list = ['accuracy', 'precision','f1', 'recall']

hyperparam_combos = list(itertools.product(l1_ratios, C_list, scorers_list))

# create a dictionary to track the error rates for different combinations
error_rates = {}

# TODO: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
for l1, C, scorer in hyperparam_combos:
    # validate the model with the different combinations of eta and lambda
    scores = logreg_variations(X, y, l1_ratio = l1, C = C, scoring = scorer)
    mean_train_score = np.mean(scores["train_score"])
    mean_test_score =  np.mean(scores["test_score"])

    if l1 == 1:
        regularizer = 'L1'
    else:
        regularizer = 'L2'
        
    print(f'With [{regularizer}, C = {C}, and scoring = {scorer}]: \n \t mean train score: {mean_train_score} \n \t mean test score: {mean_test_score}')

With [L2, C = 1.0, and scoring = accuracy]: 
 	 mean train score: 0.770652316721072 
 	 mean test score: 0.7705370151696644
With [L2, C = 1.0, and scoring = precision]: 
 	 mean train score: 0.5338897764488364 
 	 mean test score: 0.5349152141820014
With [L2, C = 1.0, and scoring = f1]: 
 	 mean train score: 0.2325448348292734 
 	 mean test score: 0.23378732645530906
With [L2, C = 1.0, and scoring = recall]: 
 	 mean train score: 0.1488938053097345 
 	 mean test score: 0.14989788972089857
With [L2, C = 0.1, and scoring = accuracy]: 
 	 mean train score: 0.7709029602225522 
 	 mean test score: 0.7702664754471729
With [L2, C = 0.1, and scoring = precision]: 
 	 mean train score: 0.538502193786322 
 	 mean test score: 0.5341717562073406
With [L2, C = 0.1, and scoring = f1]: 
 	 mean train score: 0.2228333427959468 
 	 mean test score: 0.22086948533743667
With [L2, C = 0.1, and scoring = recall]: 
 	 mean train score: 0.14063989108236896 
 	 mean test score: 0.1392784206943499
With [L2, C 

/home/ahatchett/capp-ml-30254/project-voting_demographics_project/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/ahatchett/capp-ml-30254/project-voting_demographics_project/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/ahatchett/capp-ml-30254/project-voting_demographics_project/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_d

With [L2, C = 0.0001, and scoring = precision]: 
 	 mean train score: 0.0 
 	 mean test score: 0.0
With [L2, C = 0.0001, and scoring = f1]: 
 	 mean train score: 0.0 
 	 mean test score: 0.0
With [L2, C = 0.0001, and scoring = recall]: 
 	 mean train score: 0.0 
 	 mean test score: 0.0
With [L1, C = 1.0, and scoring = accuracy]: 
 	 mean train score: 0.7706642522691126 
 	 mean test score: 0.7705051870415061
With [L1, C = 1.0, and scoring = precision]: 
 	 mean train score: 0.534045266308893 
 	 mean test score: 0.5346341101447181
With [L1, C = 1.0, and scoring = f1]: 
 	 mean train score: 0.23225385731968878 
 	 mean test score: 0.23344386546861257
With [L1, C = 1.0, and scoring = recall]: 
 	 mean train score: 0.14863852961198093 
 	 mean test score: 0.14962559564329472
With [L1, C = 0.1, and scoring = accuracy]: 
 	 mean train score: 0.7707756505592089 
 	 mean test score: 0.7707279826723289
With [L1, C = 0.1, and scoring = precision]: 
 	 mean train score: 0.5354293187739263 
 	 me

/home/ahatchett/capp-ml-30254/project-voting_demographics_project/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/ahatchett/capp-ml-30254/project-voting_demographics_project/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/ahatchett/capp-ml-30254/project-voting_demographics_project/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_d

With [L1, C = 0.001, and scoring = precision]: 
 	 mean train score: 0.0 
 	 mean test score: 0.0
With [L1, C = 0.001, and scoring = f1]: 
 	 mean train score: 0.0 
 	 mean test score: 0.0
With [L1, C = 0.001, and scoring = recall]: 
 	 mean train score: 0.0 
 	 mean test score: 0.0
With [L1, C = 0.0001, and scoring = accuracy]: 
 	 mean train score: 0.7662242591840112 
 	 mean test score: 0.7662242588509802


/home/ahatchett/capp-ml-30254/project-voting_demographics_project/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/ahatchett/capp-ml-30254/project-voting_demographics_project/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/ahatchett/capp-ml-30254/project-voting_demographics_project/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_d

With [L1, C = 0.0001, and scoring = precision]: 
 	 mean train score: 0.0 
 	 mean test score: 0.0
With [L1, C = 0.0001, and scoring = f1]: 
 	 mean train score: 0.0 
 	 mean test score: 0.0
With [L1, C = 0.0001, and scoring = recall]: 
 	 mean train score: 0.0 
 	 mean test score: 0.0


Normalization

We don't have any continuous features, so we can skip this for now.

In [ ]:
from sklearn import preprocessing
continous_features = []

# normalize all the continous features
for feature in continous_features:
    x_array = np.array(data[feature])
    normalized = preprocessing.normalize([x_array])
    data[feature] = normalized[0]

# creating new X and y training data with the normalized columns
X_norm = data[feature_order]
norm = data['VOTED']
X_train_norm, X_test_norm, y_train_norm, y_test_norm = train_test_split(X, y, test_size=0.2, random_state=1001)
normalized_scores = logreg_variations(X_train_norm, y_train_norm)

Evaluation with a class imbalance

In [26]:
# count number of not voted / voted entries
not_voted = y.sum()
voted = len(data) - not_voted

# get the proportion
prop_not_voted = (not_voted / len(data)) * 100
prop_voted = (voted / len(data)) * 100
print(f'{prop_not_voted}% of our observations did not vote, and {prop_voted}% did vote')

23.37757407937872% of our observations did not vote, and 76.62242592062128% did vote


In [38]:
f1 = logreg_variations(X_train, y_train, scoring = 'f1')
recall = logreg_variations(X_train, y_train, scoring = 'recall')
precision = recall = logreg_variations(X_train, y_train, scoring = 'precision')

print("f1:", f1)
print("recall:", recall)
print("precision:", precision)

f1: {'fit_time': array([0.06805348, 0.09462094, 0.06472921, 0.06056571, 0.06575894]), 'score_time': array([0.00445914, 0.00466585, 0.00514793, 0.00573993, 0.01407838]), 'test_score': array([0.24368114, 0.1997926 , 0.22797579, 0.22938582, 0.22484808]), 'train_score': array([0.25512195, 0.22376439, 0.22755196, 0.2114545 , 0.21799983])}
recall: {'fit_time': array([0.05359507, 0.05251288, 0.05098271, 0.05342031, 0.05278587]), 'score_time': array([0.00480151, 0.00488448, 0.00490069, 0.00502944, 0.00448084]), 'test_score': array([0.50673854, 0.52641166, 0.53895072, 0.56146179, 0.53970827]), 'train_score': array([0.53714481, 0.54247025, 0.53972056, 0.53625706, 0.53297387])}
precision: {'fit_time': array([0.05359507, 0.05251288, 0.05098271, 0.05342031, 0.05278587]), 'score_time': array([0.00480151, 0.00488448, 0.00490069, 0.00502944, 0.00448084]), 'test_score': array([0.50673854, 0.52641166, 0.53895072, 0.56146179, 0.53970827]), 'train_score': array([0.53714481, 0.54247025, 0.53972056, 0.53625